# Embedding Data Preparation (V2)

Same loading and deduplication as V1; `nlp_text` is built from original values only, for a multilingual embedding model.

In [1]:
# Analysis with extended filters

import pandas as pd, json, numpy as np
from collections import Counter
from collections import defaultdict
from pyrosm import OSM

custom_filter = {
    "amenity": [
        "restaurant", "cafe", "fast_food", "bar", "pub", "nightclub",
        "food_court", "ice_cream", "biergarten",
        "bank", "atm", "bureau_de_change", "post_office", "fuel",
        "car_wash", "car_rental", "charging_station",
        "pharmacy", "hospital", "clinic", "doctors", "dentist", "veterinary",
        "school", "university", "college", "kindergarten", "library",
        "place_of_worship", "townhall", "community_centre", "courthouse",
        "cinema", "theatre", "arts_centre", "casino", "events_venue",
        "marketplace",
        "bus_station", "taxi", "police", "fire_station", "social_facility",
    ],
    "shop": True,   
    "tourism": [
        "hotel", "hostel", "motel", "guest_house", "apartment", "camp_site",
        "caravan_site", "chalet", "wilderness_hut", "alpine_hut",
        "museum", "gallery", "attraction", "viewpoint", "theme_park", "zoo",
        "aquarium", "picnic_site",
    ],
    "leisure": [
        "park", "garden", "playground", "dog_park", "nature_reserve",
        "sports_centre", "fitness_centre", "swimming_pool", "stadium",
        "bowling_alley", "water_park", "golf_course", "ice_rink", "marina",
    ],
    "historic": ["castle", "fort", "monument", "memorial", "archaeological_site"],
    "healthcare": ["laboratory", "physiotherapist", "hospice", "alternative"],
    "office": ["company", "government", "estate_agent", "lawyer", "insurance",
               "telecommunication", "accountant", "architect", "ngo",
               "employment_agency", "tax_advisor"],
    "craft": ["tailor", "shoemaker", "carpenter", "electrician", "photographer"],
    "natural": ["peak", "spring", "cave_entrance"],
    "waterway": ["waterfall"],
    "public_transport": ["station"],
    "railway": ["station", "halt"],
    "aeroway": ["aerodrome", "terminal"],
}

extra = ["ruins", "religion", "denomination", "cuisine", "brand", "operator",
         "opening_hours", "name:en", "alt_name", "addr:street",
         "addr:housenumber", "addr:city", "wheelchair", "website", "phone"]


osm = OSM("../data/raw/bih.osm.pbf")
gdf = osm.get_pois(custom_filter=custom_filter,extra_attributes=extra)
print(gdf.shape[0])


44238


In [2]:
gdf.columns.to_list()

['id',
 'lon',
 'lat',
 'tags',
 'visible',
 'version',
 'timestamp',
 'changeset',
 'addr:city',
 'addr:country',
 'addr:housenumber',
 'addr:postcode',
 'addr:place',
 'addr:street',
 'email',
 'name',
 'opening_hours',
 'operator',
 'phone',
 'ref',
 'url',
 'website',
 'amenity',
 'atm',
 'bar',
 'bbq',
 'building',
 'building:levels',
 'cafe',
 'car_wash',
 'drinking_water',
 'fast_food',
 'fountain',
 'fuel',
 'gambling',
 'ice_cream',
 'internet_access',
 'landuse',
 'office',
 'parking',
 'post_office',
 'restaurant',
 'school',
 'social_facility',
 'source',
 'start_date',
 'wikipedia',
 'agrarian',
 'alcohol',
 'appliance',
 'bicycle',
 'books',
 'butcher',
 'clothes',
 'craft',
 'fireplace',
 'hairdresser',
 'organic',
 'outdoor',
 'religion',
 'second_hand',
 'shoes',
 'shop',
 'trade',
 'vacuum_cleaner',
 'water',
 'wholesale',
 'attraction',
 'camp_site',
 'guest_house',
 'information',
 'museum',
 'tourism',
 'leisure',
 'outdoor_seating',
 'playground',
 'archaeological

## 1. Deduplication

In the EDA, duplicates were found by `lat`/`lon`. Here we compare **geometry** instead, because polygons (13,248 rows) have no `lat`/`lon`, so duplicates among them were invisible. Location alone is not enough, though: the most common OSM duplicate is the same object mapped once as a point and once as a polygon, so steps 1.3–1.6 add criteria that do not require equal geometry.

Procedure:
1. **Same geometry + identical content** → keep one row. Content means `tags` + all filled columns, excluding metadata (`id`, `timestamp`, `version`, `changeset`, `visible`, `osm_type`, `match_check`). `tags` are compared as a dictionary, not as a string.
2. **Same geometry + compatible content** → merge. No shared key has a different value after converting Cyrillic to Latin and lowercasing (e.g. `Јањач` / `Janjač`). The kept row receives the attributes it was missing. Rows with the same geometry but different content (e.g. ARIA and Telecom Centar in the same building) stay separate.
3. **Point inside polygon + same name + same category** → merge into the point.
4. **Same `wikidata` id, within 1 km, not a brand id** → merge.
5. **Point within 25 m outside a polygon + same name + same category** → merge into the point.
6. **Shared phone or website + within 50 m + same category + equal or contained name** → merge.
7. **Same content within 10 m** → report only, for manual review; nothing is dropped.

Unless stated otherwise, the row with the highest `version`, then the newest `timestamp`, is kept. The ids of dropped rows are stored in `merged_ids`.

In [3]:
import geopandas as gpd
from geoir.utils import build_attrs, parse_tags, SKIP

# metadata: describes the OSM record, not the object itself, so it is not compared
META = SKIP | {"match_check"}
HELPER = {"geom_key", "content_key", "merged_ids"}   # helper columns created in this step
CONTENT_COLS = [c for c in gdf.columns if c not in META | HELPER]

def clean_attrs(row):
    """tags + filled columns, without metadata; strings stripped of surrounding whitespace."""
    d = {}
    for k, v in build_attrs(row).items():
        if k in META or k in HELPER:
            continue
        if isinstance(v, np.generic):
            v = v.item()
        if isinstance(v, str):
            v = v.strip()
            if not v:
                continue
        d[k] = v
    return d

def content_key(row):
    # sort_keys: the key order inside the tags JSON must not make a difference
    return json.dumps(clean_attrs(row), sort_keys=True, ensure_ascii=False, default=str)

# normalize(): the same polygon written from a different start vertex/direction gets the same key
gdf["geom_key"] = gdf.geometry.normalize().to_wkb()
gdf["content_key"] = gdf.apply(content_key, axis=1)
gdf["merged_ids"] = [[] for _ in range(len(gdf))]

# which row is kept in a group: most edited (version), then newest (timestamp)
gdf = gdf.sort_values(["version", "timestamp"], ascending=False)

same_geom = gdf.duplicated("geom_key", keep=False)
print(f"Rows sharing geometry: {same_geom.sum()} in {gdf.loc[same_geom, 'geom_key'].nunique()} groups")
print(gdf.loc[same_geom].geometry.geom_type.value_counts().to_string())

Rows sharing geometry: 24 in 12 groups
Point      22
Polygon     2


### 1.1 Same geometry, identical content → keep one row

In [4]:
n_before = len(gdf)
keys = ["geom_key", "content_key"]
first_id = gdf.groupby(keys, sort=False)["id"].transform("first")   # first = highest version, then newest
dropped = gdf[gdf["id"] != first_id]

exact_groups = gdf[gdf.duplicated(keys, keep=False)].assign(
    group=lambda x: x.groupby(keys, sort=False).ngroup(),
    keep=lambda x: x["id"] == first_id[x.index],
)
print(f"Identical groups: {exact_groups['group'].nunique()}, dropping {len(dropped)} rows")
display(exact_groups[["group", "id", "keep", "version", "timestamp", "osm_type",
                      "name", "amenity", "shop", "natural", "tourism"]]
        .sort_values(["group", "keep"], ascending=[True, False]))

merged = dropped.groupby(first_id[dropped.index])["id"].agg(list)
gdf = gdf[gdf["id"] == first_id].copy()
gdf["merged_ids"] = [m + merged.get(i, []) for i, m in zip(gdf["id"], gdf["merged_ids"])]
print(f"{n_before} -> {len(gdf)} rows")

Identical groups: 9, dropping 9 rows


,group,id,keep,version,timestamp,osm_type,name,amenity,shop,natural,tourism
4599,0,1945411050,True,3,1752107744,node,Veliki Cicibor,NaN,NaN,peak,NaN
27890,0,12990672696,False,1,1752107744,node,Veliki Cicibor,NaN,NaN,peak,NaN
30375,1,14062219002,True,1,1785435621,node,NaN,cafe,NaN,NaN,NaN
30374,1,14062210401,False,1,1785435188,node,NaN,cafe,NaN,NaN,NaN
30015,2,13913344849,True,1,1780703383,node,Lovreć,NaN,NaN,peak,NaN
30017,2,13913377997,False,1,1780702581,node,Lovreć,NaN,NaN,peak,NaN
29664,3,13724245387,True,1,1775946472,node,NaN,NaN,NaN,NaN,viewpoint
29662,3,13722520163,False,1,1775889295,node,NaN,NaN,NaN,NaN,viewpoint
29665,4,13724245388,True,1,1775946472,node,Voda Luka,NaN,NaN,spring,NaN
29663,4,13722520164,False,1,1775889295,node,Voda Luka,NaN,NaN,spring,NaN


44238 -> 44229 rows


### 1.2 Same geometry, compatible content → merge

Rows are merged when no shared attribute differs after normalization (Cyrillic → Latin, lowercase, surrounding whitespace removed). An attribute present in only one row does not count as a difference. Typos, missing diacritics (`č` vs `c`), quotes and abbreviations do count as differences, so such rows stay separate. The remaining groups are kept apart, and the attributes they differ in are printed.

In [5]:
CYR = dict(zip("абвгдђежзијклљмнњопрстћуфхцчџш",
               ["a", "b", "v", "g", "d", "đ", "e", "ž", "z", "i", "j", "k", "l", "lj", "m",
                "n", "nj", "o", "p", "r", "s", "t", "ć", "u", "f", "h", "c", "č", "dž", "š"]))

def norm_val(v):
    # Cyrillic -> Latin + lowercase, so that 'Јањач' and 'Janjač' are the same value
    if isinstance(v, str):
        return "".join(CYR.get(ch, ch) for ch in v.strip().casefold())
    return v

def compatible(a, b):
    # no shared key has a different value
    return all(norm_val(a[k]) == norm_val(b[k]) for k in a.keys() & b.keys())

def merge_rows(keep, drops, reason, verbose=True):
    """Merge rows `drops` into row `keep` (index labels): fill missing columns and tags,
    record the dropped ids in merged_ids, then remove the dropped rows from gdf."""
    global gdf
    for d in drops:
        if verbose:
            print(f"[{reason}] merging id {gdf.at[d, 'id']} ({gdf.at[d, 'name']}) "
                  f"into id {gdf.at[keep, 'id']} ({gdf.at[keep, 'name']})")
        # fill empty columns of the kept row from the dropped row
        for c in CONTENT_COLS:
            if pd.isna(gdf.at[keep, c]) and not pd.isna(gdf.at[d, c]):
                gdf.at[keep, c] = gdf.at[d, c]
        # merge tags; on a shared key the kept row's value wins
        tags = {**parse_tags(gdf.at[d, "tags"]), **parse_tags(gdf.at[keep, "tags"])}
        gdf.at[keep, "tags"] = json.dumps(tags, ensure_ascii=False) if tags else None
        gdf.at[keep, "merged_ids"] = gdf.at[keep, "merged_ids"] + [gdf.at[d, "id"]] + gdf.at[d, "merged_ids"]
    gdf = gdf.drop(index=drops)

same_geom = gdf.duplicated("geom_key", keep=False)
merges, kept_apart = [], []
for _, grp in gdf[same_geom].groupby("geom_key", sort=False):   # gdf is already sorted by version, timestamp
    clusters = []
    for idx, row in grp.iterrows():
        a = clean_attrs(row)
        for cl in clusters:
            if compatible(cl["attrs"], a):
                cl["attrs"] = {**a, **cl["attrs"]}   # values of the kept row take precedence
                cl["drop"].append(idx)
                break
        else:
            clusters.append({"keep": idx, "attrs": a, "drop": []})
    merges += [cl for cl in clusters if cl["drop"]]
    if len(clusters) > 1:
        kept_apart.append(grp)

for cl in merges:
    merge_rows(cl["keep"], cl["drop"], "same geometry")

print(f"\nSame geometry, different content (kept separate): {len(kept_apart)} groups")
for grp in kept_apart:
    a, b = (clean_attrs(r) for _, r in grp.head(2).iterrows())
    diff = sorted(k for k in a.keys() & b.keys() if norm_val(a[k]) != norm_val(b[k]))
    print(f"  id {grp['id'].tolist()}  name {grp['name'].tolist()}  differ in: {diff}")
print(f"\nTotal rows: {len(gdf)}")

[same geometry] merging id 1983711902 (Janjač) into id 1644665074 (Јањач)

Same geometry, different content (kept separate): 2 groups
  id [7909276377, 14288818672]  name ['ARIA', 'Telecom Centar Sarajevo City']  differ in: ['name', 'opening_hours', 'shop', 'website']


  id [3567237597, 2024884165]  name [nan, nan]  differ in: ['shop']

Total rows: 44228


### 1.3 Point inside polygon, same name and category → merge

The most common OSM duplicate is one object mapped twice: as a point (node) and as an area (building or site outline). Their geometries are never equal, so step 1.1 cannot find them.

A point and a polygon are merged when the point lies inside the polygon **and** both have the same normalized name **and** the same primary category. Both conditions are needed:
- the same name alone is not enough, e.g. the *Bingo* supermarket inside the *Bingo* mall, or the *Vrelo Bosne* spring inside the *Vrelo Bosne* park;
- the same category alone is not enough, e.g. two unnamed cafés in the same building.

The **point** is kept because it has the precise location and usually more attributes; the polygon fills in what is missing. A polygon that contains several matching points is ambiguous, and pairs with the same name but a different category are only listed in `pip_review` for later review.

In [6]:
CAT_KEYS = ["amenity", "shop", "tourism", "leisure", "historic", "healthcare", "office",
            "craft", "natural", "waterway", "railway", "public_transport", "aeroway"]

def text_key(s):
    """Normalized text for comparison; missing values become '' so they never match."""
    return s.map(lambda v: norm_val(v) if isinstance(v, str) else "")

def primary_category(df):
    # value of the first filled category key, in priority order; "yes" says nothing about the type
    return df[CAT_KEYS].replace("yes", np.nan).bfill(axis=1).iloc[:, 0]

gtype = gdf.geometry.geom_type
pairs = (gpd.sjoin(gdf.loc[gtype == "Point", ["geometry"]],
                   gdf.loc[gtype.isin(["Polygon", "MultiPolygon"]), ["geometry"]], predicate="within")
         .reset_index(names="pt").rename(columns={"index_right": "poly"})[["pt", "poly"]])

name_k, cat_k = text_key(gdf["name"]), text_key(primary_category(gdf))
for side in ("pt", "poly"):
    pairs[f"id_{side}"] = gdf.loc[pairs[side], "id"].to_numpy()
    pairs[f"name_{side}"] = gdf.loc[pairs[side], "name"].to_numpy()
    pairs[f"cat_{side}"] = cat_k.loc[pairs[side]].to_numpy()
same_name = (name_k.loc[pairs["pt"]].to_numpy() == name_k.loc[pairs["poly"]].to_numpy()) & (pairs["name_pt"].notna().to_numpy())
same_cat = (pairs["cat_pt"] == pairs["cat_poly"]) & (pairs["cat_pt"] != "")

match = pairs[same_name & same_cat]
ambiguous = match.duplicated("poly", keep=False)   # several matching points in one polygon -> unclear which one
to_merge = match[~ambiguous]
for pt, grp in to_merge.groupby("pt"):
    merge_rows(pt, grp["poly"].tolist(), "point in polygon", verbose=False)   # keep the point

cols = ["id_pt", "id_poly", "name_pt", "name_poly", "cat_pt", "cat_poly"]
pip_review = pd.concat([match[ambiguous][cols].assign(reason="several points in polygon"),
                        pairs[same_name & ~same_cat][cols].assign(reason="same name, different category")])
print(f"Point-in-polygon pairs: {len(pairs)}")
print(f"  same name + same category: {len(match)}  -> merged: {len(to_merge)}, ambiguous: {ambiguous.sum()}")
print(f"  same name, different category (review): {(same_name & ~same_cat).sum()}")
print(f"Total rows: {len(gdf)}")
display(to_merge[cols].head(20))
pip_review

Point-in-polygon pairs: 4604
  same name + same category: 58  -> merged: 51, ambiguous: 7
  same name, different category (review): 57
Total rows: 44177


,id_pt,id_poly,name_pt,name_poly,cat_pt,cat_poly
74,559952410,111253400,Muzej Republike Srpske,Muzej Republike Srpske,museum,museum
97,524787616,922195660,Pošte Srpske,Pošte Srpske,post_office,post_office
307,2190833609,1175570755,"Pošte Srpske, 78250 Laktaši","Pošte Srpske, 78250 Laktaši",post_office,post_office
334,2030224274,221059780,Bingo,Bingo,supermarket,supermarket
389,5314988499,1422601505,Dying Lion,Dying Lion,memorial,memorial
393,3802703579,849501760,Nešković,Nešković,fuel,fuel
398,7212593698,772536712,Apartman GM,Apartman GM,apartment,apartment
439,2190831109,480676318,Nestro,Nestro,fuel,fuel
480,1708601264,756238405,Ambulanta Tilava,Ambulanta Tilava,clinic,clinic
493,2064604725,202808757,"Triglav, Tehnički pregled vozila","Triglav, Tehnički pregled vozila",car_repair,car_repair


,id_pt,id_poly,name_pt,name_poly,cat_pt,cat_poly,reason
1867,10893658905,392873896,AC Boras,AC Boras,car,car,several points in polygon
1869,10893658907,392873896,AC Boras,AC Boras,car,car,several points in polygon
1872,9970975406,394732932,Guma M,Guma M,car,car,several points in polygon
1873,9970975407,394732932,Guma M,Guma M,car,car,several points in polygon
1876,11091945666,394732932,Guma M,Guma M,car,car,several points in polygon
...,...,...,...,...,...,...,...
4156,7400960054,168183264,Prodex,Prodex,cafe,mall,"same name, different category"
4214,6163473741,658134217,Nostalgija,Nostalgija,cafe,biergarten,"same name, different category"
4260,5736681996,8425673,Morića Han,Morića Han,restaurant,mall,"same name, different category"
4496,2041086286,172385980,Flora,Flora,restaurant,cafe,"same name, different category"


### 1.4 Shared `wikidata` id → merge

`wikidata` (e.g. `Q3490665`) identifies one real-world object, so two rows with the same id describe the same thing even when their names differ (*Vječna vatra* / *Vječna Vatra memorial*). There are two exceptions:
- **Brands:** chains such as Telemach carry the brand's id on every store. Ids that also appear as `brand:wikidata` are skipped.
- **Mistagging:** the same id on objects kilometers apart (e.g. a bike service and a bike park). Rows are merged only when they are within 1 km of each other. The distance is measured between geometries, so a point inside a large polygon has distance 0.

A point is kept over a polygon. Groups that are not merged appear in the table as `kept separate`.

In [7]:
import itertools

WD_MAX_M = 1000   # rows sharing a wikidata id further apart than this are left alone

tags_dict = gdf["tags"].map(parse_tags)
wikidata = tags_dict.map(lambda d: d.get("wikidata"))
# a wikidata id that is also used as brand:wikidata identifies a chain, not one object
brand_ids = set(tags_dict.map(lambda d: d.get("brand:wikidata")).dropna())
wikidata = wikidata[wikidata.notna() & ~wikidata.isin(brand_ids)]
shared = wikidata[wikidata.duplicated(keep=False)]

utm = gdf.geometry.to_crs(32634)   # UTM 34N, meters
is_point = gdf.geometry.geom_type == "Point"
report = []
for qid, idx in shared.groupby(shared).groups.items():
    # geometry distance: 0 when a point lies inside the polygon, unlike the distance between centroids
    max_d = max(a.distance(b) for a, b in itertools.combinations(utm.loc[idx], 2))
    decision = "merged" if max_d <= WD_MAX_M else "kept separate"
    report.append({"wikidata": qid, "ids": gdf.loc[idx, "id"].tolist(), "names": gdf.loc[idx, "name"].tolist(),
                   "osm_type": gdf.loc[idx, "osm_type"].tolist(), "max_dist_m": round(max_d), "decision": decision})
    if decision == "merged":
        # keep a point if there is one (gdf is already sorted by version, timestamp)
        keep = next((i for i in idx if is_point[i]), idx[0])
        merge_rows(keep, [i for i in idx if i != keep], "wikidata", verbose=False)

print(f"Wikidata ids excluded as brands: {len(brand_ids)}")
print(f"Groups sharing a wikidata id: {len(report)}")
print(f"Total rows: {len(gdf)}")
pd.DataFrame(report).sort_values("max_dist_m")

Wikidata ids excluded as brands: 190
Groups sharing a wikidata id: 9
Total rows: 44169


,wikidata,ids,names,osm_type,max_dist_m,decision
0,Q12981899,"[126210754, 1399701603]","[Štrbački buk, Štrbački buk]","[way, node]",0,merged
7,Q64066244,"[692745607, 5568657138]","[Park Ravne 2, Park “Ravne 2”]","[way, node]",0,merged
5,Q43951249,"[1208624885, 11041800145]","[Zelena pećina, Zelena pećina]","[way, node]",0,merged
4,Q3490665,"[1588411957, 19467025]","[Vječna vatra, Vječna Vatra memorial]","[node, relation]",0,merged
3,Q2535058,"[2625909495, 16338534]","[Vrelo Bune, Buna Wellspring, protected spring...","[node, relation]",3,merged
1,Q137646857,"[20030892, 10933440915]","[Biciklistički park Evolution, Biciklistički p...","[relation, node]",17,merged
2,Q192050,"[5300914876, 11096822459]","[nan, The monument to the victims of the Saraj...","[node, node]",18,merged
6,Q60751261,"[11924948877, 11924948878]","[Floded Trebišnjica Spring, Floded Trebišnjica...","[node, node]",576,merged
8,Q7696320,"[7190961667, 10223345464, 10223345463, 1022334...","[Telemach, Telemach, Telemach, Telemach]","[node, node, node, node]",63554,kept separate


### 1.5 Point near polygon, same name and category → merge

An extension of 1.3. A point is often placed at the entrance or on the street, just **outside** its building outline (*Radon Plaza Hotel* 15 m, *Pijaca Grbavica* 25 m), so the inside test misses it.

When a point lies outside a polygon but within 25 m of it, and both have the same normalized name and the same primary category, the polygon is merged into the point. The same-name rule makes this safe: of about 6,800 point–polygon pairs this close, only 10 match, and all are real duplicates. A point or polygon involved in more than one match is marked ambiguous and not merged.

In [8]:
NEAR_POLY_M = 25   # how far outside its polygon a point may lie

utm = gdf.geometry.to_crs(32634)   # UTM 34N, meters
gtype = gdf.geometry.geom_type
polys_buf = gpd.GeoDataFrame(geometry=utm[gtype.isin(["Polygon", "MultiPolygon"])].buffer(NEAR_POLY_M))
near = (gpd.sjoin(gpd.GeoDataFrame(geometry=utm[gtype == "Point"]), polys_buf, predicate="within")
        .reset_index(names="pt").rename(columns={"index_right": "poly"})[["pt", "poly"]])
near["dist_m"] = [utm[p].distance(utm[q]) for p, q in zip(near["pt"], near["poly"])]
# points inside the polygon were handled in 1.3
near = near[~np.array([utm[p].within(utm[q]) for p, q in zip(near["pt"], near["poly"])], dtype=bool)]

name_k, cat_k = text_key(gdf["name"]), text_key(primary_category(gdf))
same_name = (name_k.loc[near["pt"]].to_numpy() == name_k.loc[near["poly"]].to_numpy()) & (name_k.loc[near["pt"]].to_numpy() != "")
same_cat = (cat_k.loc[near["pt"]].to_numpy() == cat_k.loc[near["poly"]].to_numpy()) & (cat_k.loc[near["pt"]].to_numpy() != "")
match = near[same_name & same_cat].copy()
ambiguous = match.duplicated("poly", keep=False) | match.duplicated("pt", keep=False)
match["id_pt"] = gdf.loc[match["pt"], "id"].to_numpy()
match["id_poly"] = gdf.loc[match["poly"], "id"].to_numpy()
match["name"] = gdf.loc[match["pt"], "name"].to_numpy()
match["category"] = cat_k.loc[match["pt"]].to_numpy()
match["decision"] = np.where(ambiguous, "ambiguous (review)", "merged")

for pt, poly in zip(match.loc[~ambiguous, "pt"], match.loc[~ambiguous, "poly"]):
    merge_rows(pt, [poly], "point near polygon", verbose=False)   # keep the point

print(f"Points outside a polygon but within {NEAR_POLY_M} m of it: {len(near)}")
print(f"  same name + same category: {len(match)}  -> merged: {(~ambiguous).sum()}, ambiguous: {ambiguous.sum()}")
print(f"Total rows: {len(gdf)}")
match[["id_pt", "id_poly", "name", "category", "dist_m", "decision"]].sort_values("dist_m").round(1)

Points outside a polygon but within 25 m of it: 6798
  same name + same category: 10  -> merged: 10, ambiguous: 0
Total rows: 44159


,id_pt,id_poly,name,category,dist_m,decision
10722,3216262118,315534737,STUDIO 1,restaurant,0.0,merged
6209,4054699588,223241764,Studio Borak,photographer,1.3,merged
450,1300694703,200070310,Vatrogasno društvo Tarčin,fire_station,2.0,merged
1013,5626862829,1284682025,Džamija Slatina,place_of_worship,3.9,merged
10214,5533775695,577183352,ToGo,cafe,4.8,merged
5198,5148545010,530116004,Općinski sud u Sarajevu,courthouse,5.5,merged
1056,7438334990,17632419,Tvrđava Kruševac,archaeological_site,7.5,merged
5012,8976666737,970038819,GEA,cafe,8.8,merged
112,673355097,850107288,Radon Plaza Hotel,hotel,14.9,merged
2159,2005529316,191046576,Pijaca Grbavica,marketplace,24.6,merged


### 1.6 Shared phone or website + strict name rule → merge

On its own, a shared phone number or website is **not** a duplicate signal:
- far apart, it marks a chain (Petrol, UniCredit, Gazprom; one phone number is shared by 123 rows);
- close together, it usually marks *related* objects: a hotel and its restaurant (*Ibis Styles* / *Igman Rooftop Bistro*), a mall and its shops (*Šipad Doboj*), a bank and its ATM, two police units in one building.

It works only together with strict extra conditions. Two rows are merged when **all** of these hold:
- they share a normalized phone number or website (a value shared by more than 30 rows is ignored as a hotline);
- they are within 50 m of each other;
- they have the same primary category;
- the names are equal, or one contains the other, after normalization (*Toplik* / *Restoran Toplik*);
- the names contain the same numbers, so *Gazprom Tuzla 1* / *Gazprom Tuzla 2* and *Moja apoteka Boska* / *Moja apoteka Boska 2* stay separate.

A point is kept over a polygon.

In [9]:
import re

CONTACT_MAX_M = 50

def norm_phone(p):
    # digits only, without the country code and leading zeros: '+387 33 123-456' -> '33123456'
    d = re.sub(r"\D", "", p)
    d = re.sub(r"^(00387|387)", "", d).lstrip("0")
    return d if len(d) >= 7 else None

def norm_web(w):
    # 'https://www.Example.ba/' -> 'example.ba'
    w = re.sub(r"^(https?://)?(www\.)?", "", w.strip().lower())
    return w.rstrip("/") or None

def contact_values(i):
    """All normalized phone numbers and websites of row i (columns + tags)."""
    tags = parse_tags(gdf.at[i, "tags"])
    out = set()
    for key in ["phone", "contact:phone", "contact:mobile", "mobile"]:
        v = gdf.at[i, key] if key in gdf.columns else tags.get(key)
        if isinstance(v, str):
            out |= {("phone", n) for part in re.split(r"[;,/]", v) if (n := norm_phone(part))}
    for key in ["website", "contact:website", "url"]:
        v = gdf.at[i, key] if key in gdf.columns else tags.get(key)
        if isinstance(v, str) and (n := norm_web(v)):
            out.add(("website", n))
    return out

def similar_names(a, b):
    # equal or one contained in the other, and the same numbers ('Gazprom Tuzla 1' != 'Gazprom Tuzla 2')
    if not a or not b or not (a in b or b in a):
        return False
    return re.findall(r"\d+", a) == re.findall(r"\d+", b)

contacts = pd.DataFrame([(i, kind, val) for i in gdf.index for kind, val in contact_values(i)],
                        columns=["idx", "kind", "value"])
shared = contacts[contacts.duplicated("value", keep=False)]
print(f"Rows with phone or website: {contacts['idx'].nunique()}, values shared by several rows: {shared['value'].nunique()}")

utm = gdf.geometry.to_crs(32634)
name_k, cat_k = text_key(gdf["name"]), text_key(primary_category(gdf))
cand = []
for value, idx in shared.groupby("value")["idx"].apply(list).items():
    if len(idx) > 30:   # central hotline of a chain, not a duplicate signal
        continue
    for a, b in itertools.combinations(sorted(set(idx)), 2):
        d = utm[a].distance(utm[b])
        if d <= CONTACT_MAX_M and cat_k[a] == cat_k[b] != "" and similar_names(name_k[a], name_k[b]):
            cand.append((a, b, value, d))
cand = pd.DataFrame(cand, columns=["a", "b", "value", "dist_m"]).drop_duplicates(["a", "b"])
cand = cand.assign(id_a=gdf.loc[cand["a"], "id"].to_numpy(), id_b=gdf.loc[cand["b"], "id"].to_numpy(),
                   name_a=gdf.loc[cand["a"], "name"].to_numpy(), name_b=gdf.loc[cand["b"], "name"].to_numpy(),
                   category=cat_k.loc[cand["a"]].to_numpy())

is_point = gdf.geometry.geom_type == "Point"
n_merged = 0
for a, b in zip(cand["a"], cand["b"]):
    if a not in gdf.index or b not in gdf.index:   # already merged through another pair
        continue
    n_merged += 1
    # keep a point over a polygon, otherwise the higher version / newer row (gdf order)
    keep, drop = (a, b) if (is_point[a], -gdf.index.get_loc(a)) >= (is_point[b], -gdf.index.get_loc(b)) else (b, a)
    merge_rows(keep, [drop], "shared contact", verbose=False)

print(f"Candidate pairs: {len(cand)}, merged: {n_merged}")
print(f"Total rows: {len(gdf)}")
cand[["id_a", "id_b", "name_a", "name_b", "category", "value", "dist_m"]].sort_values("dist_m").round(1)

Rows with phone or website: 4985, values shared by several rows: 347


Candidate pairs: 10, merged: 10
Total rows: 44149


,id_a,id_b,name_a,name_b,category,value,dist_m
1,637550199,1468588079,JU Zavod za bolesti ovisnosti Kantona Sarajevo,JU Zavod za bolesti ovisnosti Kantona Sarajevo,hospital,33257750,0.0
2,194393960,680086049,Petrol,Petrol,fuel,33780091,0.0
3,6657345685,756194717,Toplik,Restoran Toplik,restaurant,57321455,0.0
11,8205375395,189256831,Cafe Bar Big Ben,Big Ben,cafe,instagram.com/caffebigben,0.0
6,10536927409,10536931041,Vulkanizer Maslo,Vulkanizer Maslo,car_repair,63284498,1.6
0,2142823971,7093893680,Pirpa,Pirpa,fast_food,33208183,15.4
4,13956950795,887531500,Heart of the nature by Armin,Armin,apartment,62114666,15.4
7,11895606009,11895606010,Don Sergio Pasticceria,Don Sergio,restaurant,66583839,15.4
5,4845221621,4944146823,Apartment Panorama Mostar,Apartment Panorama Mostar,guest_house,62833952,23.6
12,3676419925,3676419926,Kulin Dvor - ljetna bašta,Kulin Dvor,restaurant,kulindvor.com,44.0


### 1.7 Near-duplicates (review only)

Identical content with slightly shifted coordinates is not caught by comparing geometry. These pairs are **not dropped automatically**. For example, *Plivski Mlinčići* is a row of separate watermills and *Vodopad Provalije* is several waterfalls. Some of them are real duplicates, though (e.g. *Bijeli vrh* at 0.4 m, *UniCredit Bank* at 1.4 m). So `near_dups` should be reviewed manually later; a shared phone number or website (see 1.6) is a useful hint during that review.

In [10]:
# same content, named, within 10 m (but not at the same point) -> review only
NEAR_M = 10
gdf["content_key"] = gdf.apply(content_key, axis=1)   # refresh after merging
cand = gdf[gdf["name"].notna() & gdf.duplicated("content_key", keep=False)]
p = gpd.GeoDataFrame(cand[["id", "name", "content_key"]],
                     geometry=cand.geometry.representative_point(), crs=gdf.crs).to_crs(32634)  # UTM 34N, meters
pairs = gpd.sjoin(p, p.set_geometry(p.buffer(NEAR_M)), predicate="within")
pairs = pairs[(pairs["id_left"] < pairs["id_right"]) & (pairs["content_key_left"] == pairs["content_key_right"])]
pairs["dist_m"] = [p.geometry.loc[i].distance(p.geometry.loc[j]) for i, j in zip(pairs.index, pairs["index_right"])]
print(f"Pairs with identical content within {NEAR_M} m: {len(pairs)}")
near_dups = pairs[["id_left", "id_right", "name_left", "dist_m"]].sort_values("dist_m").round(1)
near_dups

Pairs with identical content within 10 m: 56


,id_left,id_right,name_left,dist_m
4901,2012695486,2325434489,Bijeli vrh,0.4
14696,7083756429,7083756430,UniCredit Bank,1.4
22135,11043594093,11525595171,MF banka,1.4
22573,11121195936,11121231235,Popov Izvor,2.2
12874,5980981955,11067755352,Vodopad Provalije,2.2
43248,1436543109,1436543114,Plivski Mlinčići,2.9
43244,1436543105,1436543106,Plivski Mlinčići,2.9
43253,1436543114,1436543115,Plivski Mlinčići,2.9
43245,1436543106,1436543107,Plivski Mlinčići,3.0
43243,1436543104,1436543108,Plivski Mlinčići,3.2


### 1.8 Preserved columns

Checks that `geometry` and `osm_type` survived deduplication.

In [11]:
assert "geometry" in gdf.columns and gdf.geometry.notna().all()
assert "osm_type" in gdf.columns
print(f"Rows after deduplication: {len(gdf)}")
print(gdf["osm_type"].value_counts().to_string())
print(gdf.geometry.geom_type.value_counts().to_string())

Rows after deduplication: 44149
osm_type
node        30974
way         13018
relation      157
Point           30974
Polygon         13116
MultiPolygon       40
LineString         19


## 2. Field Extraction (no linguistic preprocessing)

### 2.1 Attribute dictionary per row

Tags and filled columns merged into one dict per row, without metadata (same `clean_attrs` as in Section 1).

In [12]:
import geopandas as gpd
from pathlib import Path

attrs = {idx: clean_attrs(row) for idx, row in gdf.iterrows()}
print(f"Attribute dicts: {len(attrs)}")

Attribute dicts: 44149


### 2.2 Coordinates

Non-point geometries get `representative_point()` (always inside the shape); `lat`/`lon` are taken from that point.

In [13]:
is_point = gdf.geometry.geom_type == "Point"
rep_pt = gdf.geometry.where(is_point, gdf.geometry.representative_point())
gdf["lat"], gdf["lon"] = rep_pt.y, rep_pt.x
print(f"Representative points computed for {(~is_point).sum()} non-point geometries")
print(f"Missing lat/lon: {(gdf['lat'].isna() | gdf['lon'].isna()).sum()}")

Representative points computed for 13175 non-point geometries
Missing lat/lon: 0


### 2.3 Administrative enrichment

BiH admin boundaries are read from the same PBF (`osm.get_boundaries`) and joined to the points with `gpd.sjoin`; admin names are converted to Latin (bilingual names keep the Latin part, Cyrillic-only names are transliterated, `/` between Latin names becomes `,`).

In [19]:
bounds = osm.get_boundaries(boundary_type="administrative")
bounds = bounds[bounds.geometry.geom_type.isin(["Polygon", "MultiPolygon"]) & bounds["name"].notna()]
bih = bounds[(bounds["admin_level"] == "2") & bounds["name"].str.contains("Bosna", na=False)].geometry.iloc[0]
NEIGHBOURS = ("RS", "HR", "ME")

def is_foreign(tags):
    # neighbouring countries' units, clipped by the extract, can lie inside the BiH outline; their tags give them away
    t = parse_tags(tags)
    return (any(k.startswith(tuple(f"ref:{c}:" for c in NEIGHBOURS)) for k in t)
            or any(k.startswith("ISO3166") and str(v).startswith(NEIGHBOURS) for k, v in t.items()))

foreign = bounds["tags"].map(is_foreign)
print(f"Dropped foreign admin units: {sorted(bounds.loc[foreign, 'name'])}")
bounds = bounds[~foreign & bounds.geometry.representative_point().within(bih)]

ADMIN_LEVELS = {"entity": "4", "canton": "5", "city_unit": "6", "municipality": "7", "settlement": "9"}
points = gpd.GeoDataFrame(geometry=rep_pt, crs=gdf.crs)
admin = pd.DataFrame(index=gdf.index)
for col, level in ADMIN_LEVELS.items():
    polys = bounds.loc[bounds["admin_level"] == level, ["name", "geometry"]]
    joined = gpd.sjoin(points, polys, how="left", predicate="within")
    admin[col] = joined[~joined.index.duplicated()]["name"]          # overlapping polygons: keep the first
    print(f"admin_level {level:>2} ({col:12}): {len(polys):5} polygons, matched {admin[col].notna().mean():.1%} of POIs")

Dropped foreign admin units: ['Јагоштица', 'Љубовија', 'Бачевци', 'Горња Ковиљача', 'Златиборски управни округ', 'Мали Зворник', 'Оклетац', 'Општина Прибој', 'Сакар', 'Србија', 'Сјеверин', 'Централна Србија', 'Читлук']
admin_level  4 (entity      ):     3 polygons, matched 98.6% of POIs
admin_level  5 (canton      ):    10 polygons, matched 60.5% of POIs
admin_level  6 (city_unit   ):    34 polygons, matched 58.0% of POIs
admin_level  7 (municipality):   111 polygons, matched 50.6% of POIs
admin_level  9 (settlement  ):  3605 polygons, matched 72.9% of POIs


In [20]:
import re

CYR2LAT = dict(zip("абвгдђежзијклљмнњопрстћуфхцчџш",
                   ["a", "b", "v", "g", "d", "đ", "e", "ž", "z", "i", "j", "k", "l", "lj", "m",
                    "n", "nj", "o", "p", "r", "s", "t", "ć", "u", "f", "h", "c", "č", "dž", "š"]))
CYRILLIC = re.compile(r"[\u0400-\u04FF]")

def to_latin(s):
    return "".join(CYR2LAT[c.lower()].capitalize() if c.isupper() and c.lower() in CYR2LAT
                   else CYR2LAT.get(c, c) for c in s)

def latin_admin_name(name):
    # 'Општина Пале (РС) / Opština Pale (RS)' -> 'Opština Pale (RS)'; 'Лопаре' -> 'Lopare';
    # 'Srednjobosanski kanton / Županija Središnja Bosna' -> 'Srednjobosanski kanton, Županija Središnja Bosna'
    if not isinstance(name, str):
        return name
    parts = [p.strip() for p in name.split("/") if p.strip()]
    latin = [p for p in parts if not CYRILLIC.search(p)]
    return ", ".join(latin) if latin else to_latin(parts[0])

before = admin.copy()
for col in ADMIN_LEVELS:
    admin[col] = admin[col].map(latin_admin_name)

for col in ADMIN_LEVELS:
    changed = before[col].ne(admin[col]) & before[col].notna()
    examples = before.loc[changed, col].drop_duplicates().head(3)
    print(f"{col:12}: {changed.sum():6} values converted, e.g. "
          + "; ".join(f"{b!r} -> {latin_admin_name(b)!r}" for b in examples))
assert not any(admin[c].dropna().str.contains(CYRILLIC).any() for c in ADMIN_LEVELS)
assert not any(admin[c].dropna().str.contains("/", regex=False).any() for c in ADMIN_LEVELS)

entity      :  16049 values converted, e.g. 'Република Српска / Republika Srpska' -> 'Republika Srpska'
canton      :   2459 values converted, e.g. 'Srednjobosanski kanton / Županija Središnja Bosna' -> 'Srednjobosanski kanton, Županija Središnja Bosna'
city_unit   :   2063 values converted, e.g. 'Град Источно Сарајево / Grad Istočno Sarajevo' -> 'Grad Istočno Sarajevo'; 'Град Требиње / Grad Trebinje' -> 'Grad Trebinje'
municipality:   2252 values converted, e.g. 'Општина Нови Град / Opština Novi Grad' -> 'Opština Novi Grad'; 'Општина Пале (РС) / Opština Pale (RS)' -> 'Opština Pale (RS)'; 'Општина Источна Илиџа / Opština Istočna Ilidža' -> 'Opština Istočna Ilidža'
settlement  :  10145 values converted, e.g. 'Нови Град' -> 'Novi Grad'; 'Вишеград' -> 'Višegrad'; 'Влаховићи' -> 'Vlahovići'


In [21]:
# city: OSM addr:city when present, otherwise the admin unit (level 6 'Grad ...', else level 7 municipality)
osm_city = pd.Series({i: a.get("addr:city") for i, a in attrs.items()})
admin["city"] = osm_city.where(osm_city.notna(), admin["city_unit"].fillna(admin["municipality"]))
admin["city_source"] = np.where(osm_city.notna(), "osm", np.where(admin["city"].notna(), "spatial", None))
print(admin["city_source"].value_counts(dropna=False).to_string())
display(admin.sample(8, random_state=1))

city_source
spatial    36513
osm         6444
NaN         1192


,entity,canton,city_unit,municipality,settlement,city,city_source
22602,Federacija Bosne i Hercegovine,Hercegovačko-neretvanska županija (kanton),Grad Čapljina,NaN,NaN,Grad Čapljina,spatial
5554,Republika Srpska,NaN,Grad Istočno Sarajevo,Opština Istočno Novo Sarajevo,Lukavica (RS),Grad Istočno Sarajevo,spatial
4910,Federacija Bosne i Hercegovine,Kanton Sarajevo,Grad Sarajevo,Općina Centar,Sarajevo,Grad Sarajevo,spatial
3881,Federacija Bosne i Hercegovine,Zeničko-dobojski kanton,Grad Visoko,NaN,NaN,Grad Visoko,spatial
26316,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4537,Republika Srpska,NaN,Grad Istočno Sarajevo,Opština Pale (RS),Pale,Grad Istočno Sarajevo,spatial
36481,Republika Srpska,NaN,NaN,Opština Ugljevik,Donja Trnova,Donja Trnova,osm
7426,Federacija Bosne i Hercegovine,Kanton Sarajevo,Grad Sarajevo,Općina Novi Grad,Sarajevo,Grad Sarajevo,spatial


### 2.4 Structured fields

Names, category, address, contact and yes/no filters are taken as original values; only `_` in category codes becomes a space.

In [22]:
NAME_KEYS = ("alt_name", "old_name", "official_name", "short_name", "loc_name", "int_name", "reg_name")
FEATURE_SKIP = {"building", "area", "indoor", "layer", "disused", "abandoned", "historic", "landuse"}
TEXT_SKIP = {"access"}   # kept in filters, but carries no meaning as a word in nlp_text
KEY_TAGS = ["cuisine", "religion", "denomination", "sport", "clothes", "memorial", "historic:period",
            "museum", "brand", "operator"]
ENUM_TAGS = {"cuisine", "religion", "denomination", "sport", "clothes", "memorial", "historic:period", "museum"}

def split_values(v):
    return [p.strip() for p in str(v).split(";") if p.strip()]

def unique(values):
    seen, out = set(), []
    for v in values:
        if v and v.casefold() not in seen:
            seen.add(v.casefold())
            out.append(v)
    return out

def names_of(a):
    name = a.get("name") if isinstance(a.get("name"), str) else None
    alias_keys = [k for k in a if k.startswith("name:")] + [k for k in NAME_KEYS if k in a]
    aliases = unique([v for k in alias_keys if isinstance(a.get(k), str) for v in split_values(a[k])])
    return name, [x for x in aliases if not name or x.casefold() != name.casefold()]

def category_of(a):
    # first CAT_KEYS tag with a real value; generic '<key>=yes' only if nothing more specific exists
    for k in CAT_KEYS:
        v = a.get(k)
        if isinstance(v, str) and v != "yes":
            return k, v.split(";")[0].strip()
    for k in CAT_KEYS:
        if a.get(k) == "yes":
            return k, "yes"
    return None, None

def filters_of(a):
    return {k: v.strip().lower() for k, v in a.items()
            if isinstance(v, str) and v.strip().lower() in ("yes", "no")
            and k not in CAT_KEYS and k.split(":")[0] not in FEATURE_SKIP}

def contact_of(a):
    phones = unique([p for k in ("phone", "contact:phone", "mobile", "contact:mobile")
                     if isinstance(a.get(k), str) for p in split_values(a[k])])
    first = lambda *keys: next((a[k] for k in keys if isinstance(a.get(k), str)), None)
    out = {"phones": phones, "website": first("website", "contact:website", "url"),
           "email": first("email", "contact:email"),
           "facebook": first("contact:facebook", "facebook"), "instagram": first("contact:instagram")}
    return {k: v for k, v in out.items() if v}

def address_parts(a, adm):
    street = " ".join(a[k] for k in ("addr:street", "addr:housenumber") if isinstance(a.get(k), str))
    return unique([street, a.get("addr:place") if isinstance(a.get("addr:place"), str) else None,
                   adm["settlement"], adm["municipality"], adm["city"],
                   a.get("addr:postcode") if isinstance(a.get("addr:postcode"), str) else None])

### 2.5 Build `nlp_text`

Comma-separated original values: name, aliases, categories, key tags, positive yes-tags, address.

In [23]:
def nlp_text_of(a, name, aliases, adm, flt):
    cats = [a[k].split(";")[0].strip() for k in CAT_KEYS if isinstance(a.get(k), str) and a[k] != "yes"]
    if not cats:
        cats = [k for k in CAT_KEYS if a.get(k) == "yes"]
    parts = [name, *aliases, *(c.replace("_", " ") for c in cats)]
    for k in KEY_TAGS:
        if isinstance(a.get(k), str):
            vals = split_values(a[k])
            parts += [v.replace("_", " ") for v in vals] if k in ENUM_TAGS else vals
    parts += [k.replace("_", " ").replace(":", " ") for k, v in flt.items() if v == "yes" and k not in TEXT_SKIP]
    parts += address_parts(a, adm)
    return ", ".join(unique([p for p in parts if isinstance(p, str) and p.strip()]))

records = []
admin_rows = admin.to_dict("index")
for idx, a in attrs.items():
    adm = {k: (v if isinstance(v, str) else None) for k, v in admin_rows[idx].items()}
    name, aliases = names_of(a)
    cat_key, cat_val = category_of(a)
    flt, contact = filters_of(a), contact_of(a)
    records.append({
        "primary_name": name,
        "aliases": aliases,
        "primary_category": None if cat_val is None else (cat_key if cat_val == "yes" else cat_val),
        "category_key": cat_key,
        "address": ", ".join(address_parts(a, adm)) or None,
        "street": a.get("addr:street"), "housenumber": a.get("addr:housenumber"), "postcode": a.get("addr:postcode"),
        **{k: adm[k] for k in ("city", "city_source", "settlement", "municipality", "canton", "entity")},
        "opening_hours": a.get("opening_hours"),
        "contact": json.dumps(contact, ensure_ascii=False) if contact else None,
        "filters": json.dumps(flt, ensure_ascii=False) if flt else None,
        "wikidata": a.get("wikidata"),
        "nlp_text": nlp_text_of(a, name, aliases, adm, flt),
    })

gdf_v2 = gpd.GeoDataFrame(
    pd.DataFrame(records, index=gdf.index).assign(
        id=gdf["id"], osm_type=gdf["osm_type"], lat=gdf["lat"], lon=gdf["lon"], merged_ids=gdf["merged_ids"]),
    geometry=gdf.geometry, crs=gdf.crs)
gdf_v2 = gdf_v2[["id", "osm_type", "primary_name", "aliases", "primary_category", "category_key", "lat", "lon",
                 "address", "street", "housenumber", "postcode", "city", "city_source", "settlement", "municipality",
                 "canton", "entity", "opening_hours", "contact", "filters", "wikidata", "merged_ids", "nlp_text",
                 "geometry"]]

n_words = gdf_v2["nlp_text"].str.split().str.len()
print(f"gdf_v2: {gdf_v2.shape[0]} rows x {gdf_v2.shape[1]} columns")
print(f"  with primary_name: {gdf_v2['primary_name'].notna().sum()}, with filters: {gdf_v2['filters'].notna().sum()}, "
      f"with contact: {gdf_v2['contact'].notna().sum()}, with city: {gdf_v2['city'].notna().sum()}")
print(f"  empty nlp_text: {(gdf_v2['nlp_text'] == '').sum()}, words: median {n_words.median():.0f}, "
      f"90% {n_words.quantile(.9):.0f}, max {n_words.max()}")
yes_keys = Counter(k for f in gdf_v2["filters"].dropna() for k, v in json.loads(f).items() if v == "yes" and k not in TEXT_SKIP)
print(f"  yes-tags written into nlp_text ({len(yes_keys)} keys), most common: {yes_keys.most_common(15)}")

gdf_v2: 44149 rows x 25 columns
  with primary_name: 30747, with filters: 5173, with contact: 5188, with city: 42957
  empty nlp_text: 0, words: median 7, 90% 13, max 116
  yes-tags written into nlp_text (330 keys), most common: [('wheelchair', 593), ('outdoor_seating', 566), ('payment:cash', 478), ('payment:credit_cards', 309), ('atm', 289), ('payment:debit_cards', 249), ('drinking_water', 242), ('internet_access', 229), ('smoking', 221), ('payment:visa', 195), ('payment:mastercard', 189), ('air_conditioning', 177), ('fuel:diesel', 161), ('indoor_seating', 159), ('toilets', 151)]


### 2.6 Inspection checkpoint

30 random `nlp_text` values for review before embedding.

In [24]:
pd.set_option("display.max_colwidth", None)
display(gdf_v2.sample(30, random_state=0)[["id", "nlp_text"]])

,id,nlp_text
42243,1274682040,"playground, Lopare, Opština Lopare"
17277,8776692715,"Addiko Bank, bank, wheelchair, Trg Krajine 2, Banja Luka, 78000"
39362,902103098,"place of worship, christian, roman catholic, Općina Neum"
35933,517044505,"bar, Šarinci, Grad Prnjavor"
40729,1067614941,"swimming pool, Općina Ravno"
41862,1210009501,"garden, Osječani Gornji, Grad Doboj"
42980,1394979574,"garden, Čivčije Osječanske, Grad Doboj"
11086,4876489621,"Biser, Бисер, bakery, Banja Luka, Grad Banja Luka"
40861,1084988624,"place of worship, christian, serbian orthodox, Popovo Polje"
42668,1349089237,"playground, Banja Luka, Grad Banja Luka"


## 3. Embeddings & Save

Encodes `"passage: " + nlp_text` with `intfloat/multilingual-e5-base` (L2-normalized) and saves the table and the vectors in the same row order.

In [25]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-base"
model = SentenceTransformer(MODEL_NAME)

pois_v2 = gdf_v2.reset_index(drop=True)
poi_embeddings = model.encode(["passage: " + t for t in pois_v2["nlp_text"]], batch_size=64,
                              normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
print(f"Embeddings: {poi_embeddings.shape}, norms: {np.linalg.norm(poi_embeddings[:5], axis=1).round(4)}")

c:\Users\Korisnik\GeoIR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Korisnik\GeoIR\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Korisnik\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate devel

Embeddings: (44149, 768), norms: [1. 1. 1. 1. 1.]


In [26]:
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
pois_v2.to_parquet(OUT_DIR / "pois_v2.parquet")
np.save(OUT_DIR / "poi_embeddings.npy", poi_embeddings)

check = gpd.read_parquet(OUT_DIR / "pois_v2.parquet")
emb_check = np.load(OUT_DIR / "poi_embeddings.npy")
assert len(check) == emb_check.shape[0] and (check["id"].to_numpy() == pois_v2["id"].to_numpy()).all()
print(f"Saved pois_v2.parquet {check.shape} and poi_embeddings.npy {emb_check.shape}")

Saved pois_v2.parquet (44149, 25) and poi_embeddings.npy (44149, 768)
